# Ансамбль: какие модели стоит смешивать (~15 мин)
Каждая найденная на Drive модель предсказывает одну и ту же валидацию (те же 600 последовательностей `valid_sub`, что во всех прошлых запусках). Скрипт считает по официальной метрике:
- скор каждой модели отдельно;
- корреляцию прогнозов между моделями (чем ниже, тем больше смысла смешивать);
- смеси с равными весами;
- смесь с весами, подобранными на одной половине последовательностей и проверенными на другой — это **честная** оценка выигрыша.

Перед запуском:
1. Загрузите обновлённую папку `day2` на Drive в `wunder/` (новый файл `ensemble_eval.py`).
2. Положите `best2.pt` (модель сокомандника) в `wunder/` на Drive.
3. **Среда выполнения → Сменить среду выполнения → T4 GPU.**

## 1. GPU + Drive + код

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
from google.colab import drive
drive.mount('/content/drive')
W = '/content/drive/MyDrive/wunder'
!pip -q install onnx
!rm -rf /content/day2 && cp -r $W/day2 /content/day2
%cd /content/day2

## 2. Валидация (~1.5 мин)

In [ ]:
!python prepare_data.py --src $W/colab_data/valid_sub.parquet --out /content/data/valid

## 3. Все модели на одной валидации + смеси
Скрипт берёт те модели, которые найдёт на Drive; отсутствующие просто пропускаются. Для справки, скоры на лидерборде: `baseline` (×0.3) — 0.6505, `ft_last` — 0.6508, `gru_3500` — 0.6437, `mamba` — 0.643.

In [ ]:
import os
cands = {
    'baseline':   W + '/baseline.onnx',                                 # organizers' model
    'ft_last':    W + '/runs/ft_baseline_full/last.pt',                 # fine-tuned on all data
    'ft_swa':     '+'.join(W + f'/runs/ft_baseline_full/epoch_{k}.pt' for k in (2, 3, 4, 5)),  # weight average
    'gru_3500':   W + '/runs/gru128x2_v1/best.pt',
    'gru_full':   W + '/runs/gru_full/best.pt',
    'mamba':      W + '/runs/mamba_v1/best.pt',
    'team_best2': W + '/best2.pt',                                      # teammate: GRU + 8 engineered features
}
found = {k: v for k, v in cands.items() if all(os.path.exists(p) for p in v.split('+'))}
print('models found:', list(found), '| missing:', [k for k in cands if k not in found])
MODELS = ' '.join(f'--model {k}={v}' for k, v in found.items())
!python ensemble_eval.py --valid-dir /content/data/valid --out /content/preds $MODELS --report $W/ensemble_eval.json

## Как читать результат
- **Скоры отдельных моделей** сравнимы между собой (одни и те же 600 последовательностей). На лидерборде у всех наших моделей выходило на ~0.011–0.015 меньше.
- **Корреляция:** модели с корреляцией ~0.95 и выше почти не дополняют друг друга; ~0.8–0.9 — хорошие кандидаты.
- **Главная строка — `cross-validated blend` против `best single`.** Если смесь выигрывает от ~0.003, собираем ансамбль в один сабмит (две GRU в одном ONNX-графе почти не добавляют времени). Пришлите мне весь вывод ячейки 3 — отчёт также сохранён в `wunder/ensemble_eval.json`.